# NB03 — Hessen/Darmstadt Inference

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `dummyirl/darmstadt-dop20` — Hessen DOP20 patches (RGBI)

**What this does:**
1. Env setup + clone `HarishDeepak/SegEarth-OV-3`
2. Run inference on Hessen DOP20 patches
   - `cls_hessen.txt` (6 classes, enriched multi-synonym prompts)
   - `slide_crop=256` / `stride=128` (Gaussian-weighted, 50% overlap)
   - First 3 bands only (DOP20 is RGBI)
3. Save prediction masks as `.npy` + RGB overlay PNGs
4. (Optional) OSM pseudo-GT F1 if OSM masks available

> Evaluation is always **patch-based**. Never stitch predictions into tiles for metrics.

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /kaggle/working/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/kaggle/working/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/kaggle/working/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX → 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/kaggle/working/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated → {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/SegEarth-OV-3.git", str(REPO)],
        check=True)
    print(f"Cloned → {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Run Hessen inference

Config: `cls_hessen.txt` (6-class, enriched synonyms), `slide_crop=256`, `stride=128`.
The Gaussian sliding window is already wired into `segearthov3_segmentor.py`.

In [ ]:
%%bash
export MPLBACKEND=Agg
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np, os, time
os.environ["MPLBACKEND"] = "Agg"
torch.manual_seed(42); np.random.seed(42)

from pathlib import Path
from PIL import Image
from torchvision import transforms
from mmseg.structures import SegDataSample
from segearthov3_segmentor import SegEarthOV3Segmentation

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

INPUT_FOLDER = "/kaggle/input/darmstadt-dop20/"
TILE_EXTS = [".tif", ".tiff", ".png", ".jpg", ".jpeg"]
base = Path(INPUT_FOLDER)
image_paths = sorted(p for ext in TILE_EXTS for p in base.glob(f"*{ext}"))
if not image_paths:
    image_paths = sorted(p for ext in TILE_EXTS for p in base.glob(f"**/*{ext}"))
if not image_paths:
    print(f"No images found in {INPUT_FOLDER}. Contents:")
    for p in sorted(base.iterdir())[:20]: print(" ", p)
    raise SystemExit(1)
print(f"Found {len(image_paths)} patch(es)")

CLASS_NAMES = ["agricultural field", "forest", "building", "road", "water body", "background"]
COLOR_MAP = np.array([
    [210, 180, 140], [34, 139, 34], [220, 20, 60],
    [105, 105, 105], [30, 144, 255], [200, 200, 200],
], dtype=np.uint8)

model = SegEarthOV3Segmentation(
    type="SegEarthOV3Segmentation",
    classname_path="./configs/cls_hessen.txt",
    prob_thd=0.1, bg_idx=5, confidence_threshold=0.1,
    slide_stride=128, slide_crop=256,
)

os.makedirs("output", exist_ok=True)
os.makedirs("output/preds", exist_ok=True)

for idx, img_path in enumerate(tqdm(image_paths, desc="Hessen inference"), 1):
    t0 = time.time()
    img = Image.open(img_path).convert("RGB")  # drop NIR band
    img_tensor = transforms.ToTensor()(img).unsqueeze(0).to("cuda")
    ds = SegDataSample()
    ds.set_metainfo({"img_path": str(img_path), "ori_shape": img.size[::-1]})
    result = model.predict(img_tensor, data_samples=[ds])
    seg_pred = result[0].pred_sem_seg.data.cpu().numpy().squeeze(0)
    seg_logits = result[0].seg_logits.data.cpu().numpy()

    # Save raw prediction + logits
    np.save(f"output/preds/{img_path.stem}_pred.npy", seg_pred.astype(np.int16))
    np.save(f"output/preds/{img_path.stem}_logits.npy", seg_logits.astype(np.float16))

    # Save PNG overlay
    seg_rgb = COLOR_MAP[np.clip(seg_pred, 0, len(COLOR_MAP) - 1)]
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(img); axes[0].axis("off"); axes[0].set_title("RGB (Hessen DOP20 20cm GSD)")
    axes[1].imshow(img); axes[1].imshow(seg_rgb, alpha=0.45)
    axes[1].axis("off"); axes[1].set_title("Prediction — cls_hessen, Gaussian window")
    fig.legend(
        handles=[Patch(facecolor=c/255.0, label=n) for n, c in zip(CLASS_NAMES, COLOR_MAP)],
        loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.02)
    )
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig(f"output/{img_path.stem}_hessen.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [{idx}/{len(image_paths)}] {img_path.name} → {time.time()-t0:.1f}s")

print(f"Done. {len(image_paths)} patches processed.")
print(f"Predictions: output/preds/*.npy")
print(f"Overlays:    output/*.png")
EOF

## 4 — OSM pseudo-GT F1 (optional)

Requires OSM masks to be available at `/kaggle/working/osm_masks.npz`.
Generate them first using `segearth_utils.osm_eval.build_osm_masks()`
or download the pre-built file from HF if available.

Always report as **"OSM pseudo-GT F1"** — not ground-truth F1.

In [ ]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import numpy as np
from pathlib import Path

OSM_MASKS_PATH = Path("/kaggle/working/osm_masks.npz")
PREDS_DIR      = Path("output/preds")

if not OSM_MASKS_PATH.exists():
    print("OSM masks not found — skipping F1 eval.")
    print("Generate with: segearth_utils.osm_eval.build_osm_masks()")
    raise SystemExit(0)

osm_data = np.load(str(OSM_MASKS_PATH))
osm_masks = {k: osm_data[k].astype(np.int64) for k in osm_data.files}

pred_files = sorted(PREDS_DIR.glob("*_pred.npy"))
pred_masks = {p.stem.replace("_pred", ""): np.load(str(p)).astype(np.int64)
              for p in pred_files}

CLASS_NAMES = ["agricultural field", "forest", "building", "road", "water body", "background"]

# compute per-class F1
stems = sorted(set(pred_masks) & set(osm_masks))
print(f"Evaluating {len(stems)} patches...")

n_cls = 6
ignore_id = 255
tp = np.zeros(n_cls); fp = np.zeros(n_cls); fn = np.zeros(n_cls)
for stem in stems:
    pred = pred_masks[stem]
    gt   = osm_masks[stem]
    valid = gt != ignore_id
    for cls in range(n_cls):
        pc = (pred == cls) & valid
        lc = (gt   == cls) & valid
        tp[cls] += (pc & lc).sum()
        fp[cls] += (pc & ~lc).sum()
        fn[cls] += (~pc & lc).sum()

prec = tp / (tp + fp + 1e-6)
rec  = tp / (tp + fn + 1e-6)
f1   = 2 * prec * rec / (prec + rec + 1e-6)

print("\n=== OSM pseudo-GT F1 ===")
print(f"{'Class':<22} {'F1':>6}")
print("-" * 30)
for cls, (name, score) in enumerate(zip(CLASS_NAMES, f1)):
    print(f"{name:<22} {score:>6.4f}")
print("-" * 30)
print(f"{'MEAN':<22} {f1.mean():>6.4f}")
print(f"\nPatches evaluated: {len(stems)}")
EOF